In [1]:
import pandas as pd
from tqdm import tqdm
import json,re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image
import math
import shutil

def copy_file(src_path, dst_path):
    """
    Copy file from src_path to dst_path. If the destination directory does not exist, it will be created.

    Args:
        src_path (str): The path to the source file.
        dst_path (str): The path to the destination file.
    """
    # Create destination directory if it does not exist
    dst_dir = os.path.dirname(dst_path)
    if dst_dir and not os.path.exists(dst_dir):
        os.makedirs(dst_dir, exist_ok=True)
    # Copy the file
    shutil.copy2(src_path, dst_path)
    # print(f"Copied {src_path} to {dst_path}")

def read_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                json_object = json.loads(line.strip())
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
                continue
                # 如果选择抛出异常，使用下面这行
                # raise
    return data

def convert_action(action, img_width=720, img_height=1280):
    if action.startswith('Click'):
        # Extract coordinates from 'Click (x,y)' using regex
        coords = re.findall(r'\(([^)]+)', action)
        if coords:
            x, y = coords[0].split(', ')
            json_answer = {
                "action": "click",
                "coordinate": [int(float(x)*img_width),int(float(y)*img_height)]
            }
            return json_answer
            # return f"click(start_box='<|box_start|>({x},{y})<|box_end|>')"
    
    elif action == 'KEY_BACK':
        json_answer = {
            "action": "system_button",
            "button": "Back"
        }
        # return "press_back()"
        return json_answer
    
    elif action == 'KEY_HOME':
        json_answer = {
            "action": "system_button",
            "button": "Home"
        }
        return json_answer
        
    
    elif action.startswith('Stop'):
        json_answer = {
            "action": "terminate",
            "status": "success"
        }
        return json_answer

    elif action.startswith('Type'):
        # Extract text from 'Type: content'
        content = action.split(': ', 1)[-1]
        json_answer = {
            "action": "type",
            "text": content
        }
        return json_answer
        # return f"type(content='{content}')"
    
    elif action.startswith('Swipe'):
        # Extract start and end coordinates from 'Swipe (x1,y1), (x2,y2)'
        coords = re.findall(r'\(([^)]+)', action)
        if len(coords) == 2:
            x1, y1 = coords[0].split(', ')
            x2, y2 = coords[1].split(', ')
            json_answer = {
                "action": "swipe",
                "coordinate1": [int(float(x1)*img_width),int(float(y1)*img_height)],
                "coordinate2": [int(float(x2)*img_width),int(float(y2)*img_height)]
            }
            return json_answer
            # return f"scroll(start_box='<|box_start|>({x1},{y1})<|box_end|>', end_box='<|box_start|>({x2},{y2})<|box_end|>')"
    
    return "Action not recognized"

def is_gt_data(data_list):
    return all(
        data['is_correct'] == 'Y' or (data['is_correct'] == 'N' and data['human_action'])
        for data in data_list
    )

def is_trajectory_data(data_list):
    if data_list[-1]['is_correct'] == 'Y' and data_list[-1]['action'] == 'Stop':
        return True
    if data_list[-1]['is_correct'] == 'N' and data_list[-1]['human_action'] == 'Stop':
        return True
    return False

def extract_between_answer(text):
    start = text.find("<answer>") + len("<answer>")
    end = text.find("</answer>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def extract_between_think(text):
    start = text.find("<think>") + len("<think>")
    end = text.find("</think>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""


def extract_between_tool_call(text):
    start = text.find("<tool_call>") + len("<tool_call>")
    end = text.find("</tool_call>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def resize_to_multiple_of_28(width, height, max_pixels):
    # 计算当前像素总数
    current_pixels = width * height
    
    if current_pixels <= max_pixels:
        return width, height

    # 保持长宽比缩放
    aspect_ratio = width / height

    # 假设新的高度为 h，则宽度为 round(h * aspect_ratio)
    # 我们要找到最大的 h 使得 (h * w) <= max_pixels，并且 h, w 是 28 的倍数

    new_height = int((max_pixels / aspect_ratio) ** 0.5)
    new_height = (new_height // 28) * 28  # 向下取整到 28 的倍数

    new_width = int(aspect_ratio * new_height)
    new_width = (new_width // 28) * 28  # 再次确保是 28 的倍数

    # 可能由于四舍五入导致 new_width * new_height > max_pixels，再检查一次
    while new_width * new_height > max_pixels and new_height >= 28 and new_width >= 28:
        new_height -= 28
        new_width = int(aspect_ratio * new_height)
        new_width = (new_width // 28) * 28

    return new_width, new_height


def split_tool_call(text: str) -> tuple[str, str, str]:
    """
    将文本拆分成三段：
    before  : <tool_call> 之前的字符串
    middle  : <tool_call> 与 </tool_call> 之间的字符串（去掉首尾空白）
    after   : </tool_call> 之后的字符串

    如果标记不存在，则 middle 为空串，before 为原始文本，after 为空串。
    """
    open_tag, close_tag = "<tool_call>", "</tool_call>"

    start = text.find(open_tag)
    end   = text.find(close_tag, start + len(open_tag))  # 从 open_tag 之后再找，避免嵌套误匹配

    # 没找到成对标记，直接返回
    if start == -1 or end == -1:
        return text, "", ""

    before = text[:start]
    middle = text[start + len(open_tag) : end].strip()
    after  = text[end + len(close_tag) :]

    return before, middle, after
# available_gt_data_5_10_steps = [data for data in available_data_5_10_steps if is_gt_data(data['data'])]
# 读取xlsx文件
df = pd.read_excel('/home/aiqihang.aqh/GUI-R1/data/gui_test.xlsx')
column_list = df['data'].tolist()
test_instructions = [json.loads(data)['goal'].strip() for data in column_list]

image_folder = "/home/aiqihang.aqh/GUI-R1/new_images"

In [2]:
input_file_2 = "/home/aiqihang.aqh/GUI-R1/data/itag_available_agent_trajectory.jsonl"
data_2 = read_jsonl(input_file_2)

missing_id = "1920079137471119360"
max_pixels = 1003520
# scale_factor = 0.3
# missing_file_path = "/home/aiqihang.aqh/GUI-R1/new_images/63a08082-6949-42ca-8f28-5d753719e83e.png"
available_data_2 = [data for data in data_2 if data['instruction'].strip() not in test_instructions and is_gt_data(data['data']) and data['子任务包ID']!=missing_id] 

for idx, data in enumerate(available_data_2 ):
    # 对 data['data'] 按照 timestamp 升序排序
    available_data_2 [idx]['data'] = sorted(data['data'], key=lambda x: x['timestamp'])

    for i in range(len(available_data_2[idx]['data'])):

        action = available_data_2[idx]['data'][i]['action'] if available_data_2[idx]['data'][i]['is_correct'] else available_data_2[idx]['data'][i]['human_action']
        available_data_2[idx]['data'][i]['sft_action'] = convert_action(available_data_2[idx]['data'][i]['action'])
        gt_thought = available_data_2[idx]['data'][i]['human_thought'] if available_data_2[idx]['data'][i]['human_thought'] else available_data_2[idx]['data'][i]['thought']
        available_data_2[idx]['data'][i]['sft_thought'] = gt_thought

trajectory_data = [data for data in available_data_2 if is_gt_data(data['data']) and is_trajectory_data(data['data'])]
gt_but_not_trajectory_data = [data for data in available_data_2 if is_gt_data(data['data']) and not is_trajectory_data(data['data'])]

In [3]:
app2trajectory = {}

for data in trajectory_data:
    app = data['app']

    if app in app2trajectory:
        app2trajectory[app].append(data)

    else:
        app2trajectory[app] = [data]

trajectory_data_sampled = []

for key, value in app2trajectory.items():
    n = 100
    trajectory_data_sampled.extend(value[:n] if n < len(value) else value)


general_grpo_data = []


# available_data_2 = [available_data_2[0]]


system_prompt = '''You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.

You are provided with function signatures within <tools></tools> XML tags:

<tools>
{{"type": "function", "function": {{"name_for_human": "mobile_use", "name": 
"mobile_use", "description": "Use a touchscreen to interact with a mobile device."}}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:

<tool_call>
{{"name": <function-name>, "arguments": <args-json-object>}}
</tool_call>

在标签<think> </think>内输出思考过程。
在标签<tool_call> </tool_call>内输出最终答案。
'''


for data in trajectory_data_sampled :


    instruction = data['instruction']
    history_list = [item['human_subtask'] if item['human_subtask'] else item['subtask'] for item in data['data']]
    thought_list = [item['human_thought'] if item['human_thought'] else item['thought'] for item in data['data']]
    image_list = [os.path.join(image_folder, item['screenshot'].split("/")[-1]) for item in data['data']]
    action_list = [item['sft_action'] for item in data['data']]

    for step, item in enumerate(data['data']):
        if step == 0:
            history = ""
        else:
            history = "".join(history_list[:step])


        new_action = {
            "name": "mobile_use",
            "arguments": action_list[step]
        }

        user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
'''

        images = image_list[step-1:step+1] if step!=0 else [image_list[0]]
        images = [image.split("/")[-1] for image in images]
        img_list = [os.path.join(image_folder, image) for image in images]

        if len(img_list) == 1:
            user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
当前屏幕截图:
<image>
'''

        if len(img_list) == 2:
            user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
上一步屏幕截图:
<image>
当前屏幕截图:
<image>
'''      

        width, height = Image.open(img_list[-1]).size
        # width, height = int(width*scale_factor), int(height*scale_factor)
        # max_pixels = 1280*28*28
        # max_pixels = 401408
        resize_width, resize_height = resize_to_multiple_of_28(width, height, max_pixels)

        grpo_data = {
            "goal": instruction,
            "thought": item['sft_thought'],
            "action": json.dumps(new_action, ensure_ascii=False),
            "bbox": item['bbox'],
            "width": width,
            "height": height,
            "resized_width": int(resize_width),
            "resized_height": int(resize_height),
            "img_list": img_list,
            "system": system_prompt,
            "query": user_prompt
        }

        general_grpo_data.append(grpo_data)


# len(general_grpo_data)
general_grpo_data_3k = []
general_grpo_data_5k = []


for data in gt_but_not_trajectory_data :

    instruction = data['instruction']
    history_list = [item['human_subtask'] if item['human_subtask'] else item['subtask'] for item in data['data']]
    thought_list = [item['human_thought'] if item['human_thought'] else item['thought'] for item in data['data']]
    image_list = [os.path.join(image_folder, item['screenshot'].split("/")[-1]) for item in data['data']]
    action_list = [item['sft_action'] for item in data['data']]

    for step, item in enumerate(data['data']):
        if step == 0:
            history = ""
        else:
            history = "".join(history_list[:step])


        new_action = {
            "name": "mobile_use",
            "arguments": action_list[step]
        }

        user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
'''

        images = image_list[step-1:step+1] if step!=0 else [image_list[0]]
        images = [image.split("/")[-1] for image in images]
        img_list = [os.path.join(image_folder, image) for image in images]

        if len(img_list) == 1:
            user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
当前屏幕截图:
<image>
'''

        if len(img_list) == 2:
            user_prompt = f'''分析任务和历史动作，给出下一步操作。
用户任务: {instruction}
历史动作: {history}
上一步屏幕截图:
<image>
当前屏幕截图:
<image>
'''      

        width, height = Image.open(img_list[-1]).size
        # width, height = int(width*scale_factor), int(height*scale_factor)
        # max_pixels = 1280*28*28
        # max_pixels = 401408
        resize_width, resize_height = resize_to_multiple_of_28(width, height, max_pixels)

        grpo_data = {
            "goal": instruction,
            "thought": item['sft_thought'],
            "action": json.dumps(new_action, ensure_ascii=False),
            "bbox": item['bbox'],
            "width": width,
            "height": height,
            "resized_width": int(resize_width),
            "resized_height": int(resize_height),
            "img_list": img_list,
            "system": system_prompt,
            "query": user_prompt
        }

        general_grpo_data.append(grpo_data)
        if len(general_grpo_data) > 5000:
            general_grpo_data_5k = general_grpo_data
            break

        
general_grpo_data_3k = general_grpo_data_5k[:3000]
general_grpo_data_5k = general_grpo_data_5k[:5000]

In [5]:
general_grpo_data_3k[10]

{'goal': '打开小红书，搜索“点淘“， 筛选里选择排序依据”最新“，找到第一个笔记，关注笔记的作者',
 'thought': '当前已输入“点淘”作为搜索关键词，下一步要点击”搜索“按钮进入搜索结果页面。',
 'action': '{"name": "mobile_use", "arguments": {"action": "click", "coordinate": [650, 84]}}',
 'bbox': [1257, 188, 1379, 249],
 'width': 1440,
 'height': 3200,
 'resized_width': 644,
 'resized_height': 1484,
 'img_list': ['/home/aiqihang.aqh/GUI-R1/new_images/9ed045f4-a21d-48c3-badb-2f6b216fb581.png',
  '/home/aiqihang.aqh/GUI-R1/new_images/411c9b61-efe2-4740-84e6-922722ae50fc.png'],
 'system': 'You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n<tools>\n{{"type": "function", "function": {{"name_for_human": "mobile_use", "name": \n"mobile_use", "description": "Use a touchscreen to interact with a mobile device."}}}}\n</tools>\n\nFor each fu

In [6]:
import copy

general_grpo_data_3k_copy = copy.deepcopy(general_grpo_data_3k)

for data in general_grpo_data_5k:
    
    scale_actions = ['click', 'swipe']


    action = data['action']


    if type(action) != dict:

        action = json.loads(action)

    try:

        if action['arguments']['action'] in scale_actions:

            scaled_factor_x = data['resized_width']/data['width']
            scaled_factor_y = data['resized_height']/data['height']

            if action['arguments']['action'] == 'click':

                x,y = action['arguments']['coordinate']

                scaled_x, scaled_y = int(scaled_factor_x*x), int(scaled_factor_y*y)

                action['arguments']['coordinate'] = scaled_x, scaled_y

            if action['arguments']['action'] == 'swipe':

                x1,y1 = action['arguments']['coordinate1']
                scaled_x1, scaled_y1 = int(scaled_factor_x*x1), int(scaled_factor_y*y1)
                x2,y2 = action['arguments']['coordinate2']
                scaled_x2, scaled_y2 = int(scaled_factor_x*x2), int(scaled_factor_y*y2)

                action['arguments']['coordinate'] = scaled_x1, scaled_y1
                action['arguments']['coordinate2'] = scaled_x2, scaled_y2
    except:
        continue


    data['action'] = action

general_grpo_data_3k[10]

{'goal': '打开小红书，搜索“点淘“， 筛选里选择排序依据”最新“，找到第一个笔记，关注笔记的作者',
 'thought': '当前已输入“点淘”作为搜索关键词，下一步要点击”搜索“按钮进入搜索结果页面。',
 'action': {'name': 'mobile_use',
  'arguments': {'action': 'click', 'coordinate': (290, 38)}},
 'bbox': [1257, 188, 1379, 249],
 'width': 1440,
 'height': 3200,
 'resized_width': 644,
 'resized_height': 1484,
 'img_list': ['/home/aiqihang.aqh/GUI-R1/new_images/9ed045f4-a21d-48c3-badb-2f6b216fb581.png',
  '/home/aiqihang.aqh/GUI-R1/new_images/411c9b61-efe2-4740-84e6-922722ae50fc.png'],
 'system': 'You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n<tools>\n{{"type": "function", "function": {{"name_for_human": "mobile_use", "name": \n"mobile_use", "description": "Use a touchscreen to interact with a mobile device."}}}}\n</tools>\n\nFor each fu

In [9]:
all_sft_data = []

general_sft_data_3k = []

for data in general_grpo_data_3k:

    sft_data = {
        "conversations": [
            {"from": "system", "value": None},
            {"from": "human", "value": None},
            {"from": "gpt", "value": None},       
        ],
        "images": []
    }

    sft_data["images"] = data['img_list']

    sft_data["conversations"][0]['value'] = data['system']
    sft_data["conversations"][1]['value'] = data['query']
    sft_data["conversations"][2]['value'] = f"<think>\n{data['thought']}\n</think>\n<tool_call>\n{json.dumps(data['action'], ensure_ascii=False)}\n</tool_call>"

    all_sft_data.append(sft_data)
    
all_sft_data[0]

{'conversations': [{'from': 'system',
   'value': 'You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n<tools>\n{{"type": "function", "function": {{"name_for_human": "mobile_use", "name": \n"mobile_use", "description": "Use a touchscreen to interact with a mobile device."}}}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n\n<tool_call>\n{{"name": <function-name>, "arguments": <args-json-object>}}\n</tool_call>\n\n在标签<think> </think>内输出思考过程。\n在标签<tool_call> </tool_call>内输出最终答案。\n'},
  {'from': 'human',
   'value': '分析任务和历史动作，给出下一步操作。\n用户任务: 打开小红书，搜索manus，给第二个笔记点赞。\n历史动作: \n当前屏幕截图:\n<image>\n'},
  {'from': 'gpt',
   'value': '<think>\n当前在手机桌面，下一步需进入小红书应用\n</think>\n<to

In [11]:
interactive_data = read_jsonl("/home/aiqihang.aqh/Appagent/data/sft_pos.jsonl")

for idx, data in enumerate(interactive_data):
    interactive_data[idx]

    parts = interactive_data[idx]["conversations"][1]['value'].split("\n")
    parts = [part.strip() for part in parts if part]
    system_added = "\n".join(parts[1:3]) + "\n"   # 取第 2、3 行

    new_query = "\n".join(parts[:1] + parts[3:])   # parts[0] + parts[3:]
    interactive_data[idx]["conversations"][0]['value'] += system_added
    interactive_data[idx]["conversations"][1]['value'] = new_query

interactive_data[0]

{'conversations': [{'from': 'system',
   'value': 'You are a mobile GUI agent. You are given a task with the current screenshot. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n<tools>\n{{"type": "function", "function": {{"name_for_human": "mobile_use", "name": \n"mobile_use", "description": "Use a touchscreen to interact with a mobile device."}}}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n\n<tool_call>\n{{"name": <function-name>, "arguments": <args-json-object>}}\n</tool_call>在标签<think> </think>内输出思考过程。\n在标签<tool_call> </tool_call>内输出最终答案。\n'},
  {'from': 'human',
   'value': '<image>分析任务和当前图片，给出下一步操作。\n用户任务: Open WeChat, browse my Moments, check the latest updates, and recommend one to me.'},
  {'from': 'gpt',
   'value': '<think>\nThis is the WeChat login screen, which involves account login and privacy/

In [12]:
all_sft_data += interactive_data


with open(f"/home/aiqihang.aqh/Appagent/data/interactive_sft.jsonl", "w", encoding='utf-8') as file:
    for data in all_sft_data:
        json.dump(data, file, ensure_ascii=False)
        file.write("\n")

In [4]:
all_sft_data = read_jsonl("/home/aiqihang.aqh/Appagent/data/interactive_sft.jsonl")

with open("/home/aiqihang.aqh/LLaMA-Factory/data/interactive_sft.json", "w", encoding='utf-8') as file:
    json.dump(all_sft_data, file, ensure_ascii=False, indent=2)